# **DATA CLEANING STAGE**

> In this stage, all necessary screening and cleaning process will take place including;
> * Merging different search results
> * removing duplicates 
> * Standardizing data
> * Handling missing values
> * Screening to read only relevant papers

Loading all csv files into a single `Dataframe` and merge them.

In [1]:
from pathlib import Path
import pandas as pd

## path to the data files
folder_path = Path("../data/raw")

# get all CSV files in the folder
csv_files = folder_path.glob("*.csv")

dfs = [pd.read_csv(file) for file in csv_files]
print(f"Found {len(dfs)} CSV files in the folder '{folder_path}'.")

#merge all dataframes into one
df = pd.concat(dfs, ignore_index=True)

print("Total number of papers found:", len(df))
print("Columns in the dataframe:", df.columns.tolist())

Found 21 CSV files in the folder '..\data\raw'.
Total number of papers found: 525
Columns in the dataframe: ['id', 'doi', 'title', 'abstract', 'publication_year', 'publication_date', 'citation_count', 'journal', 'publisher', 'source_type', 'authors', 'first_author', 'num_authors', 'institutions', 'countries', 'keywords', 'num_keywords', 'concepts', 'top_concepts', 'referenced_works', 'open_access', 'pdf_url', 'landing_page']


## **Removing duplicates values**.

In [2]:
df = df.drop_duplicates(subset=["doi"], keep="first")
print("Total number of unique papers after removing duplicates:", len(df))


Total number of unique papers after removing duplicates: 421


## **Standardizing Data**

In [3]:
# Clean the DOI column
df["doi"] = (
    df["doi"]
    .str.lower()
    .str.replace("https://doi.org/", "", regex=False)
    .str.strip()
)

In [7]:
print(df["doi"].head())

0    10.1109/access.2021.3051315
1     10.1007/s11263-019-01228-7
2     10.1007/s10618-022-00831-6
3      10.1016/j.ins.2023.119898
4           10.1162/coli_a_00404
Name: doi, dtype: object


In [8]:
# journal names

df["journal"] = (
    df["journal"]
    .str.strip()
    .str.title()
)
print(df["journal"].head())

0                                 Ieee Access
1    International Journal Of Computer Vision
2         Data Mining And Knowledge Discovery
3                        Information Sciences
4                   Computational Linguistics
Name: journal, dtype: object


Because the journal IEEE Access was change to Ieee Acces. Let me map it for clarity and consistency.

In [9]:
journal_map = {
    "Ieee Access": "IEEE Access",
    "Automation In Construction": "Automation in Construction",
    "Engineering Structures": "Engineering Structures",
}

df["journal"] = df["journal"].replace(journal_map)
print(df["journal"].head())

0                                 IEEE Access
1    International Journal Of Computer Vision
2         Data Mining And Knowledge Discovery
3                        Information Sciences
4                   Computational Linguistics
Name: journal, dtype: object


In [10]:
# author names
df["authors"] = df["authors"].str.replace(r"\s+", " ", regex=True).str.strip()
print(df["authors"].head())

0    Ilia Stepin; José M. Alonso; Alejandro Catalá;...
1    Ramprasaath R. Selvaraju; Michael Cogswell; Ab...
2                                    Riccardo Guidotti
3    Javier Del Ser; Alejandro Barredo-Arrieta; Nat...
4     Amir Feder; Nadav Oved; Uri Shalit; Roi Reichart
Name: authors, dtype: object


In [12]:
#keywords
df["keywords"] = (
    df["keywords"]
    .str.lower()
    .str.strip()
)
print(df["keywords"].head())

0    counterfactual thinking; computer science; art...
1    closed captioning; discriminative model; convo...
2    counterfactual thinking; counterfactual condit...
3    counterfactual thinking; computer science; adv...
4    counterfactual thinking; causal model; languag...
Name: keywords, dtype: object


In [13]:
# Publication year
df["publication_year"] = df["publication_year"].astype("Int64")
print(df["publication_year"].head())

0    2021
1    2019
2    2022
3    2023
4    2021
Name: publication_year, dtype: Int64


In [14]:
print(df.isna().sum())

id                   0
doi                  0
title                0
abstract            63
publication_year     0
publication_date     0
citation_count       0
journal              0
publisher           12
source_type          0
authors              0
first_author         0
num_authors          0
institutions         5
countries            5
keywords             0
num_keywords         0
concepts             0
top_concepts        13
referenced_works     4
open_access          0
pdf_url             52
landing_page         0
dtype: int64


In [15]:
# Remove leading/trailing whitespace
text_columns = [
    "title",
    "journal",
    "publisher",
    "authors",
    "institutions",
    "countries",
    "keywords",
    "concepts",
    "top_concepts"
]

for col in text_columns:
    df[col] = df[col].str.strip()

In [16]:
# check for duplicates again after cleaning
df = df.drop_duplicates(subset=["doi"], keep="first")

In [17]:
print("Total number of unique papers after cleaning DOI:", len(df))

Total number of unique papers after cleaning DOI: 421


In [18]:
df.to_csv("../data/processed/cleaned_dataset.csv", index=False)

## **Screening of relevant papers**

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_dataset.csv")
df.head()

,id,doi,title,abstract,publication_year,publication_date,citation_count,journal,publisher,source_type,...,institutions,countries,keywords,num_keywords,concepts,top_concepts,referenced_works,open_access,pdf_url,landing_page
0,https://openalex.org/W3125997628,10.1109/access.2021.3051315,A Survey of Contrastive and Counterfactual Exp...,A number of algorithms in the field of artific...,2021,2021-01-01,364,IEEE Access,Institute of Electrical and Electronics Engineers,journal,...,Center for Research in Molecular Medicine and ...,ES,counterfactual thinking; computer science; art...,13,Counterfactual thinking; Computer science; Art...,Counterfactual thinking; Computer science; Art...,https://openalex.org/W77005467; https://openal...,True,https://ieeexplore.ieee.org/ielx7/6287639/9312...,https://doi.org/10.1109/access.2021.3051315
1,https://openalex.org/W2616247523,10.1007/s11263-019-01228-7,Grad-CAM: Visual Explanations from Deep Networ...,NaN,2019,2019-10-11,5695,International Journal Of Computer Vision,Springer Science+Business Media,journal,...,Georgia Institute of Technology; Menlo School;...,US,closed captioning; discriminative model; convo...,9,Closed captioning; Computer science; Discrimin...,Closed captioning; Computer science; Discrimin...,https://openalex.org/W1586079445; https://open...,True,https://arxiv.org/pdf/1610.02391,https://doi.org/10.1007/s11263-019-01228-7
2,https://openalex.org/W4225150645,10.1007/s10618-022-00831-6,Counterfactual explanations and how to find th...,Abstract Interpretable machine learning aims a...,2022,2022-04-28,414,Data Mining And Knowledge Discovery,Springer Science+Business Media,journal,...,University of Pisa,IT,counterfactual thinking; counterfactual condit...,16,Counterfactual thinking; Counterfactual condit...,Counterfactual thinking; Counterfactual condit...,https://openalex.org/W198744195; https://opena...,True,https://link.springer.com/content/pdf/10.1007/...,https://doi.org/10.1007/s10618-022-00831-6
3,https://openalex.org/W4388767362,10.1016/j.ins.2023.119898,On generating trustworthy counterfactual expla...,Deep learning models like chatGPT exemplify AI...,2023,2023-11-17,114,Information Sciences,Elsevier BV,journal,...,BOKU University; Instituto Andaluz de Ciencias...,AT; ES,counterfactual thinking; computer science; adv...,11,Counterfactual thinking; Computer science; Adv...,Counterfactual thinking; Computer science; Adv...,https://openalex.org/W1524926235; https://open...,True,https://doi.org/10.1016/j.ins.2023.119898,https://doi.org/10.1016/j.ins.2023.119898
4,https://openalex.org/W3031652686,10.1162/coli_a_00404,CausaLM: Causal Model Explanation Through Coun...,Abstract Understanding predictions made by dee...,2021,2021-03-29,36,Computational Linguistics,Association for Computational Linguistics,journal,...,Technion – Israel Institute of Technology,IL,counterfactual thinking; causal model; languag...,8,Counterfactual thinking; Computer science; Cau...,Counterfactual thinking; Computer science; Cau...,NaN,True,https://direct.mit.edu/coli/article-pdf/47/2/3...,https://doi.org/10.1162/coli_a_00404


Let me introduce new columns such as; "decision", "reason", "notes", and "screened".

In [3]:
new_columns = ["decision", "reason", "notes", "screened"]
for col in new_columns:
    df[col] = ""

# let all the screened be "NO" by default.
df["screened"] = "NO"

df.head()

,id,doi,title,abstract,publication_year,publication_date,citation_count,journal,publisher,source_type,...,concepts,top_concepts,referenced_works,open_access,pdf_url,landing_page,decision,reason,notes,screened
0,https://openalex.org/W3125997628,10.1109/access.2021.3051315,A Survey of Contrastive and Counterfactual Exp...,A number of algorithms in the field of artific...,2021,2021-01-01,364,IEEE Access,Institute of Electrical and Electronics Engineers,journal,...,Counterfactual thinking; Computer science; Art...,Counterfactual thinking; Computer science; Art...,https://openalex.org/W77005467; https://openal...,True,https://ieeexplore.ieee.org/ielx7/6287639/9312...,https://doi.org/10.1109/access.2021.3051315,,,,NO
1,https://openalex.org/W2616247523,10.1007/s11263-019-01228-7,Grad-CAM: Visual Explanations from Deep Networ...,NaN,2019,2019-10-11,5695,International Journal Of Computer Vision,Springer Science+Business Media,journal,...,Closed captioning; Computer science; Discrimin...,Closed captioning; Computer science; Discrimin...,https://openalex.org/W1586079445; https://open...,True,https://arxiv.org/pdf/1610.02391,https://doi.org/10.1007/s11263-019-01228-7,,,,NO
2,https://openalex.org/W4225150645,10.1007/s10618-022-00831-6,Counterfactual explanations and how to find th...,Abstract Interpretable machine learning aims a...,2022,2022-04-28,414,Data Mining And Knowledge Discovery,Springer Science+Business Media,journal,...,Counterfactual thinking; Counterfactual condit...,Counterfactual thinking; Counterfactual condit...,https://openalex.org/W198744195; https://opena...,True,https://link.springer.com/content/pdf/10.1007/...,https://doi.org/10.1007/s10618-022-00831-6,,,,NO
3,https://openalex.org/W4388767362,10.1016/j.ins.2023.119898,On generating trustworthy counterfactual expla...,Deep learning models like chatGPT exemplify AI...,2023,2023-11-17,114,Information Sciences,Elsevier BV,journal,...,Counterfactual thinking; Computer science; Adv...,Counterfactual thinking; Computer science; Adv...,https://openalex.org/W1524926235; https://open...,True,https://doi.org/10.1016/j.ins.2023.119898,https://doi.org/10.1016/j.ins.2023.119898,,,,NO
4,https://openalex.org/W3031652686,10.1162/coli_a_00404,CausaLM: Causal Model Explanation Through Coun...,Abstract Understanding predictions made by dee...,2021,2021-03-29,36,Computational Linguistics,Association for Computational Linguistics,journal,...,Counterfactual thinking; Computer science; Cau...,Counterfactual thinking; Computer science; Cau...,NaN,True,https://direct.mit.edu/coli/article-pdf/47/2/3...,https://doi.org/10.1162/coli_a_00404,,,,NO


In [8]:
df = df.drop(columns=["note"])

In [214]:
i = 100
print("Title: ")
print(df.loc[i, "title"])

print("\nAbstract: ")
print(df.loc[i, "abstract"])


Title: 
Disposable Sensors in Diagnostics, Food, and Environmental Monitoring

Abstract: 
Disposable sensors are low-cost and easy-to-use sensing devices intended for short-term or rapid single-point measurements. The growing demand for fast, accessible, and reliable information in a vastly connected world makes disposable sensors increasingly important. The areas of application for such devices are numerous, ranging from pharmaceutical, agricultural, environmental, forensic, and food sciences to wearables and clinical diagnostics, especially in resource-limited settings. The capabilities of disposable sensors can extend beyond measuring traditional physical quantities (for example, temperature or pressure); they can provide critical chemical and biological information (chemo- and biosensors) that can be digitized and made available to users and centralized/decentralized facilities for data storage, remotely. These features could pave the way for new classes of low-cost systems for hea

In [215]:
df.loc[i, "decision"] = "NO"
df.loc[i, "reason"] = "infrastructural application"
df.loc[i, "notes"] = "Healthcare domain"
df.loc[i, "screened"] = "YES"

In [216]:
df.to_csv("../data/processed/screened_dataset01.csv")

In [2]:
import pandas as pd
df = pd.read_csv("../data/processed/screened_dataset01.csv")
df.head()

,Unnamed: 0,id,doi,title,abstract,publication_year,publication_date,citation_count,journal,publisher,...,concepts,top_concepts,referenced_works,open_access,pdf_url,landing_page,decision,reason,notes,screened
0,0,https://openalex.org/W3125997628,10.1109/access.2021.3051315,A Survey of Contrastive and Counterfactual Exp...,A number of algorithms in the field of artific...,2021,2021-01-01,364,IEEE Access,Institute of Electrical and Electronics Engineers,...,Counterfactual thinking; Computer science; Art...,Counterfactual thinking; Computer science; Art...,https://openalex.org/W77005467; https://openal...,True,https://ieeexplore.ieee.org/ielx7/6287639/9312...,https://doi.org/10.1109/access.2021.3051315,NO,No infrastructure application,Purely AI with no infrastructural application,YES
1,1,https://openalex.org/W2616247523,10.1007/s11263-019-01228-7,Grad-CAM: Visual Explanations from Deep Networ...,NaN,2019,2019-10-11,5695,International Journal Of Computer Vision,Springer Science+Business Media,...,Closed captioning; Computer science; Discrimin...,Closed captioning; Computer science; Discrimin...,https://openalex.org/W1586079445; https://open...,True,https://arxiv.org/pdf/1610.02391,https://doi.org/10.1007/s11263-019-01228-7,NO,No infrastructure application,Purely AI with no infrastructural application,YES
2,2,https://openalex.org/W4225150645,10.1007/s10618-022-00831-6,Counterfactual explanations and how to find th...,Abstract Interpretable machine learning aims a...,2022,2022-04-28,414,Data Mining And Knowledge Discovery,Springer Science+Business Media,...,Counterfactual thinking; Counterfactual condit...,Counterfactual thinking; Counterfactual condit...,https://openalex.org/W198744195; https://opena...,True,https://link.springer.com/content/pdf/10.1007/...,https://doi.org/10.1007/s10618-022-00831-6,NO,No infrastructure application,Purely AI with no infrastructural application,YES
3,3,https://openalex.org/W4388767362,10.1016/j.ins.2023.119898,On generating trustworthy counterfactual expla...,Deep learning models like chatGPT exemplify AI...,2023,2023-11-17,114,Information Sciences,Elsevier BV,...,Counterfactual thinking; Computer science; Adv...,Counterfactual thinking; Computer science; Adv...,https://openalex.org/W1524926235; https://open...,True,https://doi.org/10.1016/j.ins.2023.119898,https://doi.org/10.1016/j.ins.2023.119898,NO,No infrastructure application,Purely AI with no infrastructural application,YES
4,4,https://openalex.org/W3031652686,10.1162/coli_a_00404,CausaLM: Causal Model Explanation Through Coun...,Abstract Understanding predictions made by dee...,2021,2021-03-29,36,Computational Linguistics,Association for Computational Linguistics,...,Counterfactual thinking; Computer science; Cau...,Counterfactual thinking; Computer science; Cau...,NaN,True,https://direct.mit.edu/coli/article-pdf/47/2/3...,https://doi.org/10.1162/coli_a_00404,NO,No infrastructure application,Purely AI with no infrastructural application,YES


continue screening for the next 100 papers

In [654]:
i = 420
print("Title: ")
print(df.loc[i, "title"])

print("\nAbstract: ")
print(df.loc[i, "abstract"])

Title: 
Explainable spatially explicit geospatial artificial intelligence in urban analytics

Abstract: 
Geospatial artificial intelligence (GeoAI) is proliferating in urban analytics, where graph neural networks (GNNs) have become one of the most popular methods in recent years. However, along with the success of GNNs, the black box nature of AI models has led to various concerns (e.g. algorithmic bias and model misuse) regarding their adoption in urban analytics, particularly when studying socio-economics where high transparency is a crucial component of social justice. Therefore, the desire for increased model explainability and interpretability has attracted increasing research interest. This article proposes an explainable spatially explicit GeoAI-based analytical method that combines a graph convolutional network (GCN) and a graph-based explainable AI (XAI) method, called GNNExplainer. Here, we showcase the ability of our proposed method in two studies within urban analytics: tra

In [647]:
df.loc[i, "decision"] = "NO"
df.loc[i, "reason"] = "No infrastructural application"
df.loc[i, "notes"] = "Socio-economics domain"
df.loc[i, "screened"] = "YES"

In [648]:
df['decision'].count()

np.int64(413)

In [655]:
df_scr = df[df["decision"].str.lower() == 'yes']

In [656]:
df_scr.head()

,Unnamed: 0,id,doi,title,abstract,publication_year,publication_date,citation_count,journal,publisher,...,concepts,top_concepts,referenced_works,open_access,pdf_url,landing_page,decision,reason,notes,screened
10,10,https://openalex.org/W2981731882,10.1016/j.inffus.2019.12.012,Explainable Artificial Intelligence (XAI): Con...,NaN,2019,2019-12-26,9258,Information Fusion,Elsevier BV,...,Computer science; Artificial intelligence; Tax...,Computer science; Artificial intelligence; Tax...,https://openalex.org/W9657784; https://openale...,True,https://arxiv.org/pdf/1910.10045,https://doi.org/10.1016/j.inffus.2019.12.012,YES,Explanable AI,This paper discussed the details of XAI; its c...,YES
11,11,https://openalex.org/W4288083725,10.1007/s13218-020-00637-y,One Explanation Does Not Fit All,Abstract The need for transparency of predicti...,2020,2020-02-04,166,Ki - Künstliche Intelligenz,Springer Science+Business Media,...,Transparency (behavior); Computer science; Int...,Transparency (behavior); Computer science; Int...,https://openalex.org/W1501005121; https://open...,True,https://link.springer.com/content/pdf/10.1007/...,https://doi.org/10.1007/s13218-020-00637-y,YES,Why the need of the XAI rather than heavy depe...,This paper discussed AI explanability and inte...,YES
15,15,https://openalex.org/W4386390750,10.1145/3618105,A Survey on Graph Counterfactual Explanations:...,Graph Neural Networks (GNNs) perform well in c...,2023,2023-09-02,35,Acm Computing Surveys,Association for Computing Machinery,...,Computer science; Benchmarking; Counterfactual...,Computer science; Benchmarking; Counterfactual...,https://openalex.org/W1492230849; https://open...,True,https://dl.acm.org/doi/pdf/10.1145/3618105,https://doi.org/10.1145/3618105,YES,Neural network explanations,deep learning explanability,YES
17,17,https://openalex.org/W4360992523,10.1021/acs.jctc.2c01235,A Perspective on Explanations of Molecular Pre...,Chemists can be skeptical in using deep learni...,2023,2023-03-27,106,Journal Of Chemical Theory And Computation,American Chemical Society,...,Perspective (graphical); Computer science; Dat...,Perspective (graphical); Computer science; Dat...,https://openalex.org/W330953206; https://opena...,True,https://pubs.acs.org/doi/pdf/10.1021/acs.jctc....,https://doi.org/10.1021/acs.jctc.2c01235,YES,XAI in Chemistry,this paper addressed the issue of blackbox in Ml,YES
18,18,https://openalex.org/W4384524844,10.3389/fcomp.2023.1151150,Exploring the effects of human-centered AI exp...,Transparency is widely regarded as crucial for...,2023,2023-07-17,43,Frontiers In Computer Science,Frontiers Media,...,Transparency (behavior); Heuristics; Counterfa...,Transparency (behavior); Heuristics; Counterfa...,https://openalex.org/W1491644571; https://open...,True,https://www.frontiersin.org/articles/10.3389/f...,https://doi.org/10.3389/fcomp.2023.1151150,YES,Human centered XAI,this paper addressed the issue of blackbox in Ml,YES


In [657]:
print("The number of relevant papers for this study are: ", len(df_scr))

The number of relevant papers for this study are:  135


In [660]:
df_scr.to_csv("../data/processed/fnl_scr_dataset.csv")

advanced screening